In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install --ignore-installed --force-reinstall google-adk


In [ ]:
# 1. Core ADK and Vertex AI SDK for memory support
!pip install -U google-adk google-cloud-aiplatform

# 2. Modern Google GenAI SDK (required for 2026 model compatibility)
!pip install -U google-genai


In [ ]:
!pip install bs4

In [ ]:
!pip install google-genai

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GOOGLE_API_KEY")
os.environ['GOOGLE_API_KEY'] = secret_value_0






In [ ]:
from google.adk.agents import Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

print("✅ ADK components imported successfully.")

In [ ]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1, # Initial delay before first retry (in seconds)
    http_status_codes=[429, 500, 503, 504] # Retry on these HTTP errors
)


In [ ]:
from google.adk.agents import Agent, LlmAgent
from google.adk.tools.agent_tool import AgentTool# Added LlmAgent and AgentTool to imports
from google.adk.runners import Runner
from google.adk.tools import google_search
from google.genai import types
from google.adk.sessions import InMemorySessionService
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.tools.preload_memory_tool import PreloadMemoryTool
from google.adk.memory import VertexAiMemoryBankService
from google.adk.sessions import VertexAiSessionService

In [ ]:
!gcloud auth list


In [ ]:
from pydantic import BaseModel, Field
from google.adk.tools import ToolContext
from typing import Dict, Any, List
import vertexai

class QuizQuestion(BaseModel):
    question_text: str = Field(description="The actual question content.")
    options: List[str] = Field(description="A list of 4 multiple-choice options.")
    correct_answer: str = Field(description="The exact text of the correct option.")
    explanation: str = Field(description="Rationale for the correct answer, tying to the concepts related to this question.")
    concept: str = Field(description="The underlying concept being quizzed.")
    correct_choice: str = Field(description="The letter representing the correct answer (e.g., 'A, B, C, D')")

class GeneratedQuiz(BaseModel):
    quiz_title: str = Field(description="Title of the quiz (e.g., 'AP Chem Unit 6 Quiz')")
    questions: List[QuizQuestion] = Field(description="A list of question objects.")
        
APP_NAME="google_search_agent"
USER_ID="user1234"
SESSION_ID="1234"

QUIZ_INSTRUCTIONS = """
You are the AP Chemistry Quiz Master. You MUST use tools to progress.

STRICT OPERATIONAL HIERARCHY:
1. **Identify**: If you don't know the user's name, ask. Once known, call `search_memory(name=user_name)`.
2. **Generate**: If the user asks for a quiz, you MUST:
   - Call `search_tool` with the specific AP Chem Unit/Topic.
   - You MUST extract 3-5 questions from the search results and internally save them to `generated_quiz`. 
   - DO NOT tell the user you are "finding" them; just execute the tool.
3. **Initialize**: Once questions are found, call `start_quiz()`. 
   - Display ONLY the `first_question` text and options.
4. **Evaluate**: When the user provides an answer:
   - You MUST call `submit_answer(answer="user_input")`.
   - Provide the explanation/rationale for the answer.
   - IMMEDIATELY show the next question. DO NOT ask "Ready for the next one?"; just show it.
5. **Memory**: If a user gets a question wrong, the `submit_answer` tool logs the concept.
6. **Conclude**: When `get_quiz_status` indicates all questions are answered, show the score and call it a day.
7. **Search Concepts**: If throughout this conversation the user demands concepts missed that are stored in memory, call memory_tool().
CRITICAL RULES:
- Never say "I can't generate a quiz." If you lack questions, hit the `search_tool`.
- Never give second chances on a question. One answer, one `submit_answer` call, move on.
"""

search_agent = LlmAgent(
    model="gemini-2.5-flash",
    name="search_agent",
    description="Specialist in searching for and generating academic chemistry problems.",
    instruction="""
    You are an expert chemistry researcher. When asked to provide quiz problems:
    1. Use the search tool to find specific AP Chemistry Unit 6 concepts (Thermochemistry).
    2. Format the output as a valid JSON-like object or clear text that the Quiz Agent can parse.
    3. Ensure problems include multiple-choice options and a clear correct answer.
    """,
    output_key="generated_quiz",
    output_schema=GeneratedQuiz,
    tools=[google_search] # Grounded with real search results
)


def start_quiz(tool_context: ToolContext,) -> Dict[str, Any]:
    state = tool_context.state
    
    quiz_data = state.get("generated_quiz")

    if not quiz_data:
        return {"status": "error", "error_message": "No quiz found. Ask the search_agent to create one first!"}

    questions = quiz_data.questions if hasattr(quiz_data, 'questions') else quiz_data.get("questions", [])
    
    state["quiz_questions"] = questions
    state["quiz_started"] = True
    state["current_question_index"] = 0
    state["correct_answers"] = 0
    state["total_answered"] = 0
    state["score_percentage"] = 0
    state["total_questions"] = len(questions)
    state["missed_questions"] = []
    state["missed_concepts"] = []
    
    if questions:
        return {
            "status": "started",
            "quiz_title": quiz_data.get("quiz_title"),
            "first_question": questions[0].get("question_text"), # Matches your schema field
            "question_number": 1,
            "total_questions": len(questions),
        }
        
    return {"status": "error", "error_message": "No questions available"}

def submit_answer(tool_context: ToolContext, answer: str ) -> Dict[str, any]:
    state = tool_context.state
    i = state.get("current_question_index", 0)
    questions = state.get("quiz_questions", [])
    if not questions or i >= len(questions):
        return {"error": "No active question found or quiz finished."}

    # 2. Get the specific question object
    current_q = questions[i]

    correct_answer = str(current_q.get("correct_answer")).strip()
    correct_choice = str(current_q.get("correct_choice")).strip()

    is_correct = (answer.strip() == correct_answer and correct_answer != "") or \
                 (answer.strip() == correct_choice and correct_choice != "")
    
    state["total_answered"] = state.get("total_answered", 0) + 1

    explanation = current_q.get("explanation", "No explanation provided")
    concept = current_q.get("concept", "General")

    if is_correct:
        state["correct_answers"] = state.get("correct_answers", 0) + 1
    else:
        # Save the question text for the "missed" list
        state["missed_questions"].append(current_q.get("question_text"))
        state["missed_concepts"].append(current_q.get("concept"))

    state['current_question_index'] = i + 1
    
    
    return {
        "correct": is_correct,
        "feedback": "Correct!" if is_correct else f"Wrong. The answer was {correct_answer}",
        "explanation": explanation,
        "concept": concept
    }


def get_quiz_status(tool_context: ToolContext) -> Dict[str, Any]:
    state = tool_context.state
    
    # 1. Use .get(key, 0) to avoid NoneType errors
    answered = state.get('total_answered', 0)
    total = state.get('total_questions', 0)
    correct = state.get("correct_answers", 0)
    current_idx = state.get("current_question_index", 0)

    # 2. Check for zero to prevent crash
    score = (correct / answered) if answered > 0 else 0
    
    # 3. Determine if we should trigger the Vector DB archive
    is_finished = (answered >= total) and total > 0

    response = {
        "current_progress": f"Question {current_idx + 1} of {total}",
        "score_so_far": f"{score:.2%}",
        "answered": answered,
        "is_quiz_finished": is_finished
    }

    # 4. If finished, add a hint for the agent to archive missed concepts
    if is_finished:
        response["next_steps"] = "The quiz is over. I will now archive your missed concepts to your long-term memory."
        
    return response

def reset_quiz(tool_context: ToolContext) -> Dict[str, Any]:
    state = tool_context.state
    
    # 1. Retrieve quiz data safely
    quiz_data = state.get("generated_quiz")
    if not quiz_data:
        return {"status": "error", "message": "No quiz exists to reset."}

    # Handle dictionary or object schema
    questions = quiz_data.get("questions", []) if isinstance(quiz_data, dict) else getattr(quiz_data, 'questions', [])
    
    # 2. FULL RESET of the session state
    # Critical for accurate Vector Database archiving
    state["quiz_started"] = True
    state["current_question_index"] = 0
    state["correct_answers"] = 0
    state["total_answered"] = 0
    state["missed_questions"] = [] # MUST reset this
    state["missed_concepts"] = []  # MUST reset this
    state["total_questions"] = len(questions)
    
    # 3. Synchronize with Vertex AI Session Service
    # (Optional but recommended to force a state sync)
    
    if questions:
        # Access the first question safely
        first_q = questions[0]
        q_text = first_q.get("question_text") if isinstance(first_q, dict) else getattr(first_q, "question_text", "")
        
        return {
            "status": "reset_success",
            "message": "The quiz has been reset. Here is your first question.",
            "first_question": q_text,
            "total_questions": len(questions) # Fixed the typo here
        }
    return {"status": "error", "message": "The stored quiz contains no questions."}


In [ ]:
from google.adk.tools.google_search_tool import GoogleSearchTool

user_secret = UserSecretsClient()
user_secret.set_gcloud_credentials()

project_id = UserSecretsClient().get_secret("GCP_PROJECT_ID")

memory_tool = PreloadMemoryTool()

client = vertexai.Client(
    project=project_id,
    location="us-central1",
)

agent_engine = client.agent_engines.create()
agent_engine_id = agent_engine.api_resource.name.split("/")[-1]
app_name = "your-app-name"

generator_agent = LlmAgent(
    model='gemini-2.5-flash',
    name='search_agent',
    instruction="Based off user prompt, search up relevant concepts and problems. Create a set of problems for a quiz and express them in JSON format.",
    output_schema=GeneratedQuiz,
    output_key="generated_quiz"
)

generator_tool = AgentTool(agent=generator_agent)

quiz_agent = LlmAgent(
    model="gemini-2.5-flash",
    name="tutor",
    instruction=QUIZ_INSTRUCTIONS,
    tools=[start_quiz, submit_answer, get_quiz_status, reset_quiz, memory_tool, generator_tool]
)


In [ ]:
APP_NAME = f"projects/{project_id}/locations/us-central1/reasoningEngines/{agent_engine_id}"

memory_service = VertexAiMemoryBankService(
    project=project_id,
    location="us-central1",
    agent_engine_id=agent_engine_id
)

session_service = VertexAiSessionService(
    project=project_id,
    location="us-central1",
    agent_engine_id=agent_engine_id
)

async def setup_session_and_runner():
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
    runner = Runner(agent=quiz_agent, 
                    app_name=APP_NAME, 
                    session_service=session_service, 
                    memory_service=memory_service,
                    )
    
    return session, runner


In [ ]:
import asyncio

async def run_single_turn(query, session, user_id, runner):
        """Run a single conversation turn."""
        content = types.Content(role="user", parts=[types.Part(text=query)])
        events = runner.run_async(user_id=user_id, session_id=session.id, new_message=content)

        response_content = None
        async for event in events:
            if event.is_final_response():
                response_content = event.content.parts[0].text
                
        return response_content


async def chat_loop(session, user_id, runner) -> None:
            """Main chat interface loop."""
            print("\nStarting chat. Type 'exit' or 'quit' to end.")
            print("Every message will be automatically stored in memory.\n")
        
            while True:
                user_input = input("\nYou: ")
                if user_input.lower() in ["quit", "exit", "bye"]:
                    print("\nAssistant: Thank you for chatting. Have a great day!")
                    break
        
                response = await run_single_turn(user_input, session, user_id, runner=runner)
                if response:
                    print(f"\nAssistant: {response}")
        
            completed_session = await runner.session_service.get_session(app_name=app_name, user_id=USER_ID, session_id=session.id)
            
            await memory_service.add_session_to_memory(completed_session)

if __name__ == "__main__":
    session, runner = await setup_session_and_runner()
    await chat_loop(session=session, user_id=USER_ID, runner=runner)
